Master Fact Table (transactions)
        │
        ▼
Weekly Aggregation
        │
        ▼
Regime Detection Dataset

# Imports

In [33]:
import pandas as pd
import numpy as np

financial=pd.read_csv("/content/financial_tracking (1).csv")
inventory=pd.read_csv("/content/inventory (2).csv")
sales=pd.read_csv("/content/sales_transactions (1).csv")

print("Sales shape:", sales.shape)
print("Financial shape:", financial.shape)
print("Inventory shape:", inventory.shape)

Sales shape: (5872, 24)
Financial shape: (6024, 18)
Inventory shape: (1118, 23)


In [34]:
print("sales csv columns", sales.columns)
print("financial csv columns", financial.columns)
print("inventory csv columns", inventory.columns)

sales csv columns Index(['order_id', 'order_date', 'order_time', 'order_hour', 'sku_id',
       'sku_name', 'category', 'channel', 'region', 'distributor_id',
       'qty_ordered', 'qty_delivered', 'fulfillment_rate_pct',
       'is_partial_delivery', 'unit_cost', 'selling_price_per_unit',
       'discount_pct', 'total_revenue', 'gross_margin', 'payment_status',
       'days_to_payment', 'transaction_type', 'return_reason',
       'original_order_id'],
      dtype='object')
financial csv columns Index(['transaction_id', 'date', 'sku_id', 'sku_name', 'category',
       'transaction_type', 'channel', 'region', 'quantity', 'unit_cost',
       'selling_price', 'total_value', 'gross_margin', 'discount_pct',
       'payment_status', 'days_to_payment', 'reference_id', 'damage_reason'],
      dtype='object')
inventory csv columns Index(['sku_id', 'sku_name', 'category', 'batch_no', 'supplier_id',
       'supplier_name', 'warehouse_id', 'manufactured_date', 'expiry_date',
       'shelf_life_day

In [35]:
sales.columns=sales.columns.str.lower().str.strip()
financial.columns=financial.columns.str.lower().str.strip()
inventory.columns=inventory.columns.str.lower().str.strip()




# data preprocessing

In [55]:
sales['order_date']=pd.to_datetime(sales['order_date'])
financial['date'] = pd.to_datetime(financial['date'])
inventory['last_updated'] = pd.to_datetime(inventory['last_updated'])

In [56]:
#Weekly captures structural change.

sales["week"]=sales['order_date'].dt.to_period('W')
financial["week"]=financial['date'].dt.to_period('W')
inventory["week"]=inventory['last_updated'].dt.to_period('W')

# Signals

In [57]:
sales_ts=sales.groupby('week').agg({'unit_cost':'mean',
    'total_revenue':'sum',
    'discount_pct':'mean'}).reset_index()

sales_ts['channel_count']=sales.groupby('week')['channel'].nunique().values
sales_ts['region_count'] = sales.groupby('week')['region'].nunique().values
sales_ts['sku_count'] = sales.groupby('week')['sku_id'].nunique().values

In [58]:
finance_ts=financial.groupby('week').agg({'unit_cost':'mean',
    'selling_price':'mean',
    'total_value':'sum',
    'discount_pct':'mean'}).reset_index()

finance_ts.rename(columns={
    'unit_cost':'finance_cost',
    'selling_price':'avg_price',
    'total_value':'finance_revenue',
    'discount_pct':'finance_discount'
}, inplace=True)

In [59]:
inventory_ts=inventory.groupby('week').agg({'cost_per_unit':'mean',
    'quantity_in_stock':'sum'}).reset_index()

inventory_ts.rename(columns={
    'cost_per_unit':'inventory_cost',
    'quantity_in_stock':'stock_level'
},inplace=True)

# Regime

In [60]:
#weekly regime signal dataset

regime_df=sales_ts.merge(finance_ts,on='week',how='left')
regime_df = regime_df.merge(inventory_ts, on='week', how='left')

regime_df = regime_df.sort_values('week')

In [61]:
regime_df.head()

,week,unit_cost,total_revenue,discount_pct,channel_count,region_count,sku_count,finance_cost,avg_price,finance_revenue,finance_discount,inventory_cost,stock_level
0,2025-02-17/2025-02-23,476.020370,6568449.33,4.567901,4,8,69,476.020370,454.727160,6568449.33,4.567901,NaN,NaN
1,2025-02-24/2025-03-02,499.950500,11300909.18,4.050000,4,8,93,499.950500,475.980000,11300909.18,4.050000,NaN,NaN
2,2025-03-03/2025-03-09,461.030734,11919173.74,3.623853,4,8,97,461.030734,444.697064,11919173.74,3.623853,NaN,NaN
3,2025-03-10/2025-03-16,459.084649,11685206.06,4.473684,4,8,101,459.084649,435.007281,11685206.06,4.473684,NaN,NaN
4,2025-03-17/2025-03-23,446.572549,10283159.77,4.460784,4,8,93,446.572549,424.778137,10283159.77,4.460784,NaN,NaN


In [62]:
regime_df['week_start']=regime_df['week'].apply(lambda x: x.start_time)
regime_df=regime_df.sort_values('week_start')

In [63]:
#Supply Shock Feature (Cost Change)

#Supply shocks come from cost volatility
#Supply shock if cost_pct_change > 0.15 for 3 weeks

regime_df['cost_pct_change']=regime_df['unit_cost'].pct_change()

In [64]:
'''
Cost Volatility

Rolling volatility detects unstable supply
high volatility = unstable supplier market.
'''

#Instead of looking at your entire history at once, pandas creates a "sliding window" that only looks at 4 weeks at a time.calculate the value for Week 10, the window looks at Weeks 7, 8, 9, and 10
#In a "Stable Regime," your cost_volatility_4w will be a very low, flat line. If you see this number suddenly spike, it’s a warning signal that your supply chain or market prices are becoming unstable

regime_df['cost_volatility_4w'] = regime_df['unit_cost'].rolling(4, min_periods=2).std()



In [65]:
'''
Demand Shock Feature (Revenue Drop)

Demand shocks come from revenue collapse.
Meaning revenue dropped more than 20%.
'''
regime_df['revenue_pct_change'] = regime_df['total_revenue'].pct_change()

#Revenue Volatility
regime_df['revune_volatility_4w']=regime_df['total_revenue'].rolling(4).std()



In [66]:
'''
Competitive Pressure Feature

Competitive pressure appears as discount spikes
'''

regime_df['discount_volatility_4w'] = regime_df['discount_pct'].rolling(4).std()

# Z score

In [67]:
'''
Z-score = 0: The discount this week is exactly the same as the average of the last 4 weeks. (Business as usual).

Z-score = +2.0: The discount is much higher than normal. You are giving away way more than the recent average.

Z-score = -2.0: The discount is much lower than normal. You are being much "stingier" than usual.
alculating a Rolling Z-score.

Instead of comparing today to the "all-time" average, you are comparing today to the last 4 weeks (a one-month window).
'''

regime_df['discount_volatility_4w'] = regime_df['discount_pct'].rolling(4).std()
regime_df['discount_zscore'] = (
    regime_df['discount_pct'] - regime_df['discount_pct'].rolling(4, min_periods=2).mean()
) / regime_df['discount_pct'].rolling(4, min_periods=2).std()

In [68]:
regime_df['channel_diff'] = regime_df['channel_count'].diff()
regime_df['region_diff'] = regime_df['region_count'].diff()
regime_df['sku_diff'] = regime_df['sku_count'].diff()


#baseline
regime_df['cost_ma4'] = regime_df['unit_cost'].rolling(4).mean()
regime_df['revenue_ma4'] = regime_df['total_revenue'].rolling(4).mean()
regime_df['discount_ma4'] = regime_df['discount_pct'].rolling(4).mean()

In [69]:
regime_df.shape

regime_df.head()



,week,unit_cost,total_revenue,discount_pct,channel_count,region_count,sku_count,finance_cost,avg_price,finance_revenue,...,revenue_pct_change,revune_volatility_4w,discount_volatility_4w,discount_zscore,channel_diff,region_diff,sku_diff,cost_ma4,revenue_ma4,discount_ma4
0,2025-02-17/2025-02-23,476.020370,6568449.33,4.567901,4,8,69,476.020370,454.727160,6568449.33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-02-24/2025-03-02,499.950500,11300909.18,4.050000,4,8,93,499.950500,475.980000,11300909.18,...,0.720484,NaN,NaN,-0.707107,0.0,0.0,24.0,NaN,NaN,NaN
2,2025-03-03/2025-03-09,461.030734,11919173.74,3.623853,4,8,97,461.030734,444.697064,11919173.74,...,0.054709,NaN,NaN,-0.966083,0.0,0.0,4.0,NaN,NaN,NaN
3,2025-03-10/2025-03-16,459.084649,11685206.06,4.473684,4,8,101,459.084649,435.007281,11685206.06,...,-0.019630,2.546113e+06,0.433171,0.680619,0.0,0.0,4.0,474.021563,1.036843e+07,4.17886
4,2025-03-17/2025-03-23,446.572549,10283159.77,4.460784,4,8,93,446.572549,424.778137,10283159.77,...,-0.119985,7.224245e+05,0.403390,0.765273,0.0,0.0,-8.0,466.659608,1.129711e+07,4.15208


# Handling Missing Values

In [70]:
regime_df['channel_diff'] = regime_df['channel_diff'].fillna(0)
regime_df['region_diff'] = regime_df['region_diff'].fillna(0)
regime_df['sku_diff'] = regime_df['sku_diff'].fillna(0)

regime_df['cost_pct_change'] = regime_df['cost_pct_change'].fillna(0)
regime_df['revenue_pct_change'] = regime_df['revenue_pct_change'].fillna(0)


regime_df['cost_volatility_4w'] = regime_df['cost_volatility_4w'].fillna(0)
regime_df['revune_volatility_4w'] = regime_df['revune_volatility_4w'].fillna(0)
regime_df['discount_volatility_4w'] = regime_df['discount_volatility_4w'].fillna(0)

regime_df['discount_zscore'] = regime_df['discount_zscore'].fillna(0)

regime_df['cost_ma4'] = regime_df['cost_ma4'].fillna(regime_df['unit_cost'])
regime_df['revenue_ma4'] = regime_df['revenue_ma4'].fillna(regime_df['total_revenue'])
regime_df['discount_ma4'] = regime_df['discount_ma4'].fillna(regime_df['discount_pct'])

In [71]:
regime_df.isna().sum()

,0
week,0
unit_cost,0
total_revenue,0
discount_pct,0
channel_count,0
region_count,0
sku_count,0
finance_cost,0
avg_price,0
finance_revenue,0


In [72]:
regime_df.head()

,week,unit_cost,total_revenue,discount_pct,channel_count,region_count,sku_count,finance_cost,avg_price,finance_revenue,...,revenue_pct_change,revune_volatility_4w,discount_volatility_4w,discount_zscore,channel_diff,region_diff,sku_diff,cost_ma4,revenue_ma4,discount_ma4
0,2025-02-17/2025-02-23,476.020370,6568449.33,4.567901,4,8,69,476.020370,454.727160,6568449.33,...,0.000000,0.000000e+00,0.000000,0.000000,0.0,0.0,0.0,476.020370,6.568449e+06,4.567901
1,2025-02-24/2025-03-02,499.950500,11300909.18,4.050000,4,8,93,499.950500,475.980000,11300909.18,...,0.720484,0.000000e+00,0.000000,-0.707107,0.0,0.0,24.0,499.950500,1.130091e+07,4.050000
2,2025-03-03/2025-03-09,461.030734,11919173.74,3.623853,4,8,97,461.030734,444.697064,11919173.74,...,0.054709,0.000000e+00,0.000000,-0.966083,0.0,0.0,4.0,461.030734,1.191917e+07,3.623853
3,2025-03-10/2025-03-16,459.084649,11685206.06,4.473684,4,8,101,459.084649,435.007281,11685206.06,...,-0.019630,2.546113e+06,0.433171,0.680619,0.0,0.0,4.0,474.021563,1.036843e+07,4.178860
4,2025-03-17/2025-03-23,446.572549,10283159.77,4.460784,4,8,93,446.572549,424.778137,10283159.77,...,-0.119985,7.224245e+05,0.403390,0.765273,0.0,0.0,-8.0,466.659608,1.129711e+07,4.152080


In [74]:
regime_df.to_csv("regime_detection_dataset.csv", index=False)
from google.colab import files


regime_df.to_parquet("regime_detection_dataset.parquet", index=False)